In [ ]:
import sys
# !{sys.executable} -m pip install tiktoken
# !{sys.executable} -m pip install pandas>=2.0.0
import os

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # for 1B
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import tiktoken
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig,LlamaTokenizer
from datasets import load_dataset
from peft import prepare_model_for_kbit_training
from peft import LoraConfig, get_peft_model
from datetime import datetime

In [ ]:
from trl import SFTTrainer, setup_chat_format
import json
from peft import PeftModel, PeftConfig,AutoPeftModelForCausalLM
from transformers import MllamaForConditionalGeneration, AutoProcessor
import torch
torch_dtype = torch.float16
attn_implementation = "eager"
# !{sys.executable} -m pip install bitsandbytes
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)
import time

def load_model_tokenizer_questions(base_model, lora_model):
    
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = MllamaForConditionalGeneration.from_pretrained(base_model, quantization_config=bnb_config, device_map="auto",attn_implementation=attn_implementation)
    model.resize_token_embeddings(len(tokenizer))  # Or len(tokenizer) if you're using the LoRA tokenizer
    model = PeftModel.from_pretrained(model, lora_model, device_map="auto")
    
    return model, tokenizer

In [ ]:
model, tokenizer = load_model_tokenizer_questions(
    base_model="Qwen/Qwen2.5-7B",
    lora_model="xfu20/BEMGPT_1.0.0"
)

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import numpy as np
import requests

In [ ]:
def download_github_json_files(repo_owner, repo_name, folder_path, branch="main"):
    """
    Recursively downloads all JSON (.json) files from a GitHub repository folder
    to the current directory.
    
    Parameters:
    - repo_owner: GitHub username or organization
    - repo_name: repository name
    - folder_path: path to folder in the repo
    - branch: branch name (default "main")
    """
    api_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents/{folder_path}?ref={branch}"
    r = requests.get(api_url)
    if r.status_code != 200:
        print(f"Failed to access {api_url}: {r.status_code}")
        return

    contents = r.json()
    
    for item in contents:
        item_type = item['type']
        item_name = item['name']
        item_path = item['path']
        if item_type == "file" and item_name.endswith(".json"):
            download_url = item['download_url']
            print(f"Downloading {item_path}...")
            file_resp = requests.get(download_url)
            file_path = os.path.join(os.getcwd(), item_name)  # save in current directory
            with open(file_path, "wb") as f:
                f.write(file_resp.content)
        elif item_type == "dir":
            # Recursive call for subdirectories
            download_github_json_files(repo_owner, repo_name, item_path, branch)

repo_owner = "fuArizona"
repo_name = "BEMGPT"
folder_path = "data/dataset/training/Pairs"
download_github_json_files(repo_owner, repo_name, folder_path)

In [ ]:
# ----------------------------------
# List of potential QA pair files
# ----------------------------------
qa_files = [
    "engineering-reference.json",
]

# ----------------------------------
# Load QA pairs from all existing files
# ----------------------------------
qa_pairs = []
for path in qa_files:
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, list):
                    qa_pairs.extend(data)
        except Exception as e:
            print(f"Failed to load {path}: {e}")


# ----------------------------------
# Handle empty or missing QA data
# ----------------------------------
if qa_pairs:
    questions = [pair.get("input", "").strip() for pair in qa_pairs if pair.get("input")]
    answers = [pair.get("output", "").strip() for pair in qa_pairs if pair.get("output")]
else:
    qa_pairs, questions, answers = [], [], []

# ----------------------------------
# Build FAISS retrieval index
# ----------------------------------
embed_model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and efficient
if questions:
    question_embeddings = embed_model.encode(questions, convert_to_numpy=True)
    embedding_dim = question_embeddings.shape[1]
    index = faiss.IndexFlatIP(embedding_dim)  # Inner product = cosine when normalized
    faiss.normalize_L2(question_embeddings)
    index.add(question_embeddings)
else:
    # Empty fallback index (zero-dimension) for safe querying
    index = faiss.IndexFlatIP(384)  # default dim for MiniLM

In [ ]:
def generate_with_model(prompt, model, tokenizer, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.5,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def ask(query, model, tokenizer, top_k=3, max_new_tokens=512):

    q_emb = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, top_k)

    threshold = 0.8
    base_answer = None
    if D[0][0] >= threshold:
        candidate_answer = answers[I[0][0]]
        if candidate_answer and candidate_answer.strip():
            base_answer = candidate_answer.strip()


    if base_answer:
        prompt = f"""
You are an expert in building energy simulation and EnergyPlus.
Focus only on answering the following question using the context of the original answer.
Do not include unrelated information or excessive elaboration.

**Question:** {query}

**Original Answer:** {base_answer}

**Expanded and Polished Answer (aligned with the EnergyPlus Engineering Reference):**
"""
    else:
        prompt = f"""
You are an expert in building energy simulation and EnergyPlus.
Answer the following question clearly, concisely, and accurately.
Only include information relevant to the question.

**Question:** {query}

**Answer (aligned with the EnergyPlus Engineering Reference):**
"""


    polished_answer = generate_with_model(prompt, model, tokenizer, max_new_tokens=max_new_tokens)

    # Clean output: keep only text after the headers
    marker1 = "**Expanded and Polished Answer (aligned with the EnergyPlus Engineering Reference):**"
    marker2 = "**Answer (aligned with the EnergyPlus Engineering Reference):**"
    
    if marker1 in polished_answer:
        polished_answer = polished_answer.split(marker1)[-1].strip()
    elif marker2 in polished_answer:
        polished_answer = polished_answer.split(marker2)[-1].strip()


    # Ensure the text ends with a proper sentence
    if not polished_answer.endswith(('.', '!', '?')):
        polished_answer += '.'

    return polished_answer


In [ ]:
response = ask("In EnergyPlus, what are two main types of loops within the HVAC simulation?", model, tokenizer)
print(response)

In [ ]:
response = ask("In EnergyPlus, what is the equation to calculate infiltration according to Coblenz and Achenbach in 1963?", model, tokenizer)
print(response)